In [5]:
from pathlib import Path
import json
import pandas as pd
from lineage_from_tracking import LineageFromTracking
from metrics_division import load_full_division_events, evaluate_single_prediction



In [6]:

temporal_tolerance = 3
max_future_frame_offset = 2
iou_thresh_mother = 0.5
iou_thresh_bud = 0.2
pred_mask_dir = Path("/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12")
gt_mask_dir = Path("/home/hcourtei/Projects/Cell_proj/data/moma_N_3_checked/moma/val/CTC/12_GT/TRA")


##  1. Post processing des cellules trackées pour déterminer la lignée cellulaire

In [7]:
# 1. Initialiser l'analyser
lineage_analyzer = LineageFromTracking(pred_mask_dir, prefix = 'mask')

# 2. Charger les durées de vie
min_lifetime = 5
df_lifetimes = lineage_analyzer.load_cell_lifetimes(min_lifetime=min_lifetime)
print(f"{len(df_lifetimes)} cellules conservées après filtrage")

# 3. Calculer les relations de lignée
df_lineage = lineage_analyzer.compute_lineage(
    df_lifetimes,
    max_distance=35,
    min_parent_age=1,
    enforce_parent_larger=True
)

# 4. Afficher les résultats

print(f"Nb cellules lignées prédites: {len(df_lineage)}")

# 5. Sauvegarder le résultat
lineage_from_track_path = pred_mask_dir / "summary" / "lineage_from_track.txt"
lineage_analyzer.save_lineage(lineage_from_track_path)
print(f"Résultat sauvegardé dans {lineage_from_track_path}")
print("\nPRED_TRACK par post-traitement:")
df_lineage

22 cellules conservées après filtrage
Nb cellules lignées prédites: 22
Lineage saved to /home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/lineage_from_track.txt
Résultat sauvegardé dans /home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/lineage_from_track.txt

PRED_TRACK par post-traitement:


,cell_id,begin_frame,end_frame,life_time,centroid,area,parent_id
0,1,0,88,89,"(32, 66)",89,0
1,2,11,63,53,"(31, 53)",53,1
2,3,36,124,89,"(32, 64)",89,1
3,6,38,66,29,"(31, 50)",29,2
4,9,43,55,13,"(39, 58)",13,6
5,15,52,92,41,"(36, 58)",41,1
6,17,57,63,7,"(22, 12)",7,2
7,18,61,167,107,"(33, 74)",107,3
8,19,62,66,5,"(28, 23)",5,6
9,20,84,88,5,"(28, 34)",5,15


## 2. Evaluation des prédictions de lignée

In [8]:
# Charger les événements GT
gt_events = load_full_division_events(pred_mask_dir / "summary" / "man_track.txt")
# Charger les événements prédits par celssam2
pred_events = load_full_division_events(pred_mask_dir / "summary" / "res_track.txt")
man_track_df = pd.read_csv(pred_mask_dir / "summary" / "man_track.txt", sep='\s+', header=None,
                            names=['cell_id', 'begin_frame', 'end_frame', 'parent_id'])
man_track_df.insert(3, "life_time", man_track_df["end_frame"] - man_track_df["begin_frame"] + 1)
print("MAN_TRACK.txt ")

man_track_df

MAN_TRACK.txt 


,cell_id,begin_frame,end_frame,life_time,parent_id
0,1,0,88,89,0
1,2,11,63,53,1
2,3,34,66,33,2
3,4,35,124,90,1
4,5,52,92,41,1
5,6,56,63,8,2
6,7,60,66,7,3
7,8,60,167,108,4
8,9,84,88,5,1
9,10,85,144,60,8


### a. avec Cellsam2 version asym_no_gate

In [9]:
lineag_from_cellsam2 = evaluate_single_prediction(
    gt_mask_dir=gt_mask_dir,
    man_track_path=pred_mask_dir / "summary" / "man_track.txt",
    pred_mask_dir=pred_mask_dir,
    pred_lineage_path=pred_mask_dir / "summary" / "res_track.txt",
    temporal_tolerance=3,
    max_future_frame_offset=2,
    iou_thresh_mother=0.5,
    iou_thresh_bud=0.2
)
lineag_from_cellsam2

{'gt_video_dir': '/home/hcourtei/Projects/Cell_proj/data/moma_N_3_checked/moma/val/CTC/12_GT/TRA',
 'pred_video_dir': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12',
 'man_track_path': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/man_track.txt',
 'pred_lineage_path': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/res_track.txt',
 'temporal_tolerance': 3,
 'max_future_frame_offset': 2,
 'iou_thresh_mother': 0.5,
 'iou_thresh_bud': 0.2,
 'metrics': {'gt_division_events': 20,
  'pred_division_events': 35,
  'parentless_pred': 11,
  'parentless_ratio': 0.314,
  'tp': 11,
  'fp': 24,
  'fn': 9,
  'precision': 0.314,
  'recall': 0.55,
  'f1': 0.4,
  'avg_time_error': 0.455,
  'std_time_error': 0.656,
  'avg_iou_mother': 0.947,
  'avg_iou_bud': 0.787}}

### b. en post process à partir des cellules segmentées et trackées

In [10]:
lineag_from_track = evaluate_single_prediction(
    gt_mask_dir=gt_mask_dir,
    man_track_path=pred_mask_dir / "summary" / "man_track.txt",
    pred_mask_dir=pred_mask_dir,
    pred_lineage_path=lineage_from_track_path,
    temporal_tolerance=3,
    max_future_frame_offset=2,
    iou_thresh_mother=0.5,
    iou_thresh_bud=0.2
)
lineag_from_track

{'gt_video_dir': '/home/hcourtei/Projects/Cell_proj/data/moma_N_3_checked/moma/val/CTC/12_GT/TRA',
 'pred_video_dir': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12',
 'man_track_path': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/man_track.txt',
 'pred_lineage_path': '/home/hcourtei/Projects/Cell_proj/CellSam2Gilles/eval_model/model:moma_N3_checked_v100/data_vers:moma_N3_checked/12/summary/lineage_from_track.txt',
 'temporal_tolerance': 3,
 'max_future_frame_offset': 2,
 'iou_thresh_mother': 0.5,
 'iou_thresh_bud': 0.2,
 'metrics': {'gt_division_events': 20,
  'pred_division_events': 21,
  'parentless_pred': 1,
  'parentless_ratio': 0.048,
  'tp': 11,
  'fp': 10,
  'fn': 9,
  'precision': 0.524,
  'recall': 0.55,
  'f1': 0.537,
  'avg_time_error': 0.636,
  'std_time_error': 0.643,
  'avg_iou_mother': 0.947,
  'avg_iou_bud': 0.81}}